# Phase 3 — RAG Retrieval Prototype

This notebook builds and evaluates the scientific retrieval layer.

Pipeline:

1. Load the validated scientific corpus
2. Remove empty pages
3. Clean extraction artifacts
4. Chunk scientific text
5. Generate stable chunk IDs
6. Preserve source/page metadata
7. Generate semantic embeddings
8. Store embeddings in ChromaDB
9. Implement top-k retrieval
10. Evaluate retrieval against golden cases

The retrieval system must achieve at least an 80% top-3 retrieval hit rate
before downstream LLM reasoning is added.

Our 5 PDFs
     ↓
Break them into sensible pieces
     ↓
Give every piece an ID
     ↓
Remember its PDF + page number
     ↓
MiniLM understands the meaning
     ↓
Store everything in ChromaDB
     ↓
Ask a question
     ↓
Find the 3 most relevant pieces
     ↓
Test using GC01–GC07
     ↓
At least 6/7 work correctly
     ↓
PHASE 3 COMPLETE ✅

## 1. Load and Prepare Scientific Corpus

The validated page-level scientific corpus created during Phase 2 is loaded
for retrieval preparation. Empty or non-extractable pages are excluded from
the retrieval corpus while the original processed corpus remains unchanged.

In [1]:
import pandas as pd

corpus_path = "../data/processed/scientific_corpus_pages.csv"

corpus_df = pd.read_csv(corpus_path)

print("Corpus loaded successfully.")
print("Shape:", corpus_df.shape)
print("\nColumns:")
print(corpus_df.columns.tolist())

corpus_df.head()

Corpus loaded successfully.
Shape: (1377, 4)

Columns:
['source_id', 'filename', 'page_number', 'text']


,source_id,filename,page_number,text
0,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,1,STATE of KNOWLEDGE\r\nof SOIL BIODIVERSITY\r\n...
1,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,2,NaN
2,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,3,STATE of KNOWLEDGE\r\nof SOIL BIODIVERSITY\r\n...
3,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,"Required citation\r\nFAO, ITPS, GSBI, CBD and ..."
4,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,5,III\r\nCONTENTS\r\nContributors\r\nXVI\r\nFor...


In [2]:
print("Total page records:", len(corpus_df))
print("Sources:", sorted(corpus_df["source_id"].unique()))

print("\nMissing values:")
print(corpus_df.isnull().sum())

Total page records: 1377
Sources: ['SRC01', 'SRC02', 'SRC03', 'SRC04', 'SRC05']

Missing values:
source_id       0
filename        0
page_number     0
text           25
dtype: int64


In [3]:
retrieval_df = corpus_df.copy()

# Remove pages with no extracted text
retrieval_df = retrieval_df.dropna(subset=["text"])

# Remove whitespace-only text
retrieval_df["text"] = retrieval_df["text"].astype(str).str.strip()
retrieval_df = retrieval_df[retrieval_df["text"] != ""].copy()

# Reset row numbers
retrieval_df.reset_index(drop=True, inplace=True)

In [4]:
print("Original pages:", len(corpus_df))
print("Retrievable pages:", len(retrieval_df))
print("Removed empty pages:", len(corpus_df) - len(retrieval_df))

print("\nPages remaining per source:")
print(retrieval_df.groupby("source_id").size())

print("\nRemaining null text:", retrieval_df["text"].isnull().sum())
print(
    "Remaining empty text:",
    (retrieval_df["text"].str.strip() == "").sum()
)

Original pages: 1377
Retrievable pages: 1352
Removed empty pages: 25

Pages remaining per source:
source_id
SRC01    613
SRC02    560
SRC03    166
SRC04      5
SRC05      8
dtype: int64

Remaining null text: 0
Remaining empty text: 0


## 2. Text Cleaning and Sentence-Aware Chunking

Extracted PDF text is cleaned conservatively to normalize whitespace while
preserving scientific values, units, citations, and terminology.

The cleaned text is then divided into sentence-aware chunks so that scientific
statements are not unnecessarily split in the middle of a sentence.

In [5]:
import re

def clean_text(text):
    # Replace line breaks/tabs/repeated whitespace with a single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()


retrieval_df["clean_text"] = retrieval_df["text"].apply(clean_text)

In [6]:
print("Empty cleaned text:",
      (retrieval_df["clean_text"] == "").sum())

print("\nExample BEFORE:\n")
print(retrieval_df.iloc[0]["text"][:500])

print("\nExample AFTER:\n")
print(retrieval_df.iloc[0]["clean_text"][:500])

Empty cleaned text: 0

Example BEFORE:

STATE of KNOWLEDGE
of SOIL BIODIVERSITY
Status, challenges and potentialities
Report
2020
﻿

Example AFTER:

STATE of KNOWLEDGE of SOIL BIODIVERSITY Status, challenges and potentialities Report 2020 ﻿


In [7]:
def split_into_sentences(text):
    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9])',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

In [8]:
sample_text = retrieval_df.iloc[10]["clean_text"]

sample_sentences = split_into_sentences(sample_text)

print("Number of sentences:", len(sample_sentences))

for sentence in sample_sentences[:10]:
    print("\n-", sentence)

Number of sentences: 1

- X 7.1 | Assessment of soil biodiversit 430 7.1.1 | Contributions of soil biodiversity to ecosystems services 430 7.1.2 | National assessment 430 7.1.3 | Practical applications of soil biodiversit 431 7.1.4 | Major practices negatively impacting soil biodiversit 431 7.1.5 | Invasive alien species (IAS 431 7.1.6 | Monitoring soil biodiversit 432 7.1.7 | Indicators used to evaluate soil biodiversit 432 7.2 | Research, capacity development and awareness raisin 433 7.3 | Mainstreaming: policies, programmes, regulations and governmental framework 433 7.4 | Analysis of the main gaps, barriers and opportunities in the conservation and sustainable use of soil biodiversit 434 Annex II National Survey on Status of Soil Biodiversity: Knowledge, Challenges and Opportunitie 437 I | Assessmen 439 II | Research, capacity development and awareness raisin 442 III | Mainstreaming: policies, regulations and governmental framework 442 IV | Analysis of gaps and opportunities 443 An

In [9]:
sample_row = retrieval_df[
    (retrieval_df["source_id"] == "SRC01") &
    (retrieval_df["page_number"] == 100)
].iloc[0]

sample_text = sample_row["clean_text"]

print("Source:", sample_row["source_id"])
print("Page:", sample_row["page_number"])
print("\nTEXT SAMPLE:\n")
print(sample_text[:1500])

Source: SRC01
Page: 100

TEXT SAMPLE:

State of knowledge of soil biodiversity 70 beetle abundance may be more important for ecosystem functioning than species richness (Manning and Cutler, 2018; Alvarado et al., 2019). However, functional group richness, species composition and maintenance of interactions among co-occurring species (such as tunnelling and dwelling dung beetles; Nervo et al., 2017) have been shown to be important for ecosystem function (Larsen et al., 2005; Milotić et al., 2018; Menéndez et al., 2016) and for sustaining multiple ecosystem functions (O’hea et al., 2010; Nervo et al., 2017; Manning et al., 2016; Piccini et al., 2018; Santos-Heredia et al., 2018) in both perturbed (Beynon et al., 2012) and unperturbed systems (Manning et al., 2017; Beynon et al., 2012). In addition to their direct consumptive and shredding activities, macrofauna can indirectly affect soil C dynamics by altering the abundance and activity of their prey and/ or regulating the activity or co

In [10]:
sample_sentences = split_into_sentences(sample_text)

print("Number of sentences:", len(sample_sentences))

for i, sentence in enumerate(sample_sentences[:10], start=1):
    print(f"\nSentence {i}:")
    print(sentence)

Number of sentences: 13

Sentence 1:
State of knowledge of soil biodiversity 70 beetle abundance may be more important for ecosystem functioning than species richness (Manning and Cutler, 2018; Alvarado et al., 2019).

Sentence 2:
However, functional group richness, species composition and maintenance of interactions among co-occurring species (such as tunnelling and dwelling dung beetles; Nervo et al., 2017) have been shown to be important for ecosystem function (Larsen et al., 2005; Milotić et al., 2018; Menéndez et al., 2016) and for sustaining multiple ecosystem functions (O’hea et al., 2010; Nervo et al., 2017; Manning et al., 2016; Piccini et al., 2018; Santos-Heredia et al., 2018) in both perturbed (Beynon et al., 2012) and unperturbed systems (Manning et al., 2017; Beynon et al., 2012).

Sentence 3:
In addition to their direct consumptive and shredding activities, macrofauna can indirectly affect soil C dynamics by altering the abundance and activity of their prey and/ or regul

In [14]:
def create_chunks(text, target_size=1200, overlap_sentences=2):
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    # Fallback:
    # If sentence splitting produces one extremely long block
    # (common for TOCs, lists, or badly structured PDF text),
    # split it safely by character windows.
    if len(sentences) == 1 and len(sentences[0]) > target_size:
        chunks = []
        start = 0
        overlap_chars = 200

        while start < len(text):
            end = start + target_size
            chunk = text[start:end].strip()

            if chunk:
                chunks.append(chunk)

            if end >= len(text):
                break

            start = end - overlap_chars

        return chunks

    # Normal sentence-aware chunking
    chunks = []
    current_sentences = []

    for sentence in sentences:
        candidate = " ".join(current_sentences + [sentence])

        if len(candidate) <= target_size or not current_sentences:
            current_sentences.append(sentence)

        else:
            chunk = " ".join(current_sentences).strip()

            if chunk:
                chunks.append(chunk)

            current_sentences = (
                current_sentences[-overlap_sentences:]
                if overlap_sentences > 0
                else []
            )

            current_sentences.append(sentence)

    if current_sentences:
        chunk = " ".join(current_sentences).strip()

        if chunk:
            chunks.append(chunk)

    return chunks

In [13]:
sample_chunks = create_chunks(sample_text)

print("Number of chunks:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks, start=1):
    print(f"\nCHUNK {i} — {len(chunk)} characters")
    print(chunk[:250])

Number of chunks: 6

CHUNK 1 — 1001 characters
State of knowledge of soil biodiversity 70 beetle abundance may be more important for ecosystem functioning than species richness (Manning and Cutler, 2018; Alvarado et al., 2019). However, functional group richness, species composition and maintenan

CHUNK 2 — 1073 characters
However, functional group richness, species composition and maintenance of interactions among co-occurring species (such as tunnelling and dwelling dung beetles; Nervo et al., 2017) have been shown to be important for ecosystem function (Larsen et al

CHUNK 3 — 1060 characters
In addition to their direct consumptive and shredding activities, macrofauna can indirectly affect soil C dynamics by altering the abundance and activity of their prey and/ or regulating the activity or composition of soil microbial communities. For 

CHUNK 4 — 1162 characters
Larger invertebrate predators, such as predatory beetles and spiders, can also have cascading effects through the food 

In [15]:
normal_chunks = create_chunks(sample_text)

print("Normal scientific page")
print("Number of chunks:", len(normal_chunks))

for i, chunk in enumerate(normal_chunks, 1):
    print(f"Chunk {i}: {len(chunk)} characters")

Normal scientific page
Number of chunks: 6
Chunk 1: 1001 characters
Chunk 2: 1073 characters
Chunk 3: 1060 characters
Chunk 4: 1162 characters
Chunk 5: 1196 characters
Chunk 6: 974 characters


In [16]:
toc_text = retrieval_df.iloc[10]["clean_text"]

print("Sentences detected:", len(split_into_sentences(toc_text)))

toc_chunks = create_chunks(toc_text)

print("TOC chunks:", len(toc_chunks))

for i, chunk in enumerate(toc_chunks, start=1):
    print(f"Chunk {i}: {len(chunk)} characters")

Sentences detected: 1
TOC chunks: 1
Chunk 1: 1081 characters


In [ ]:
## 3. Generate Corpus Chunks and Metadata

The validated retrieval corpus is divided into manageable chunks. Each chunk
receives a deterministic identifier and retains its original source, filename,
page number, and within-page chunk index for provenance and citation tracing.

In [17]:
all_chunks = []

for _, row in retrieval_df.iterrows():

    page_chunks = create_chunks(row["clean_text"])

    for chunk_index, chunk in enumerate(page_chunks, start=1):

        chunk_id = (
            f'{row["source_id"]}_'
            f'P{int(row["page_number"]):04d}_'
            f'C{chunk_index:03d}'
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "source_id": row["source_id"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "text": chunk
        })

In [18]:
chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
print("Columns:", chunks_df.columns.tolist())

chunks_df.head()

Total chunks: 5652
Columns: ['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text']


,chunk_id,source_id,filename,page_number,chunk_index,text
0,SRC01_P0001_C001,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,1,1,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
1,SRC01_P0003_C001,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,3,1,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
2,SRC01_P0004_C001,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,1,"Required citation FAO, ITPS, GSBI, CBD and EC...."
3,SRC01_P0004_C002,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,2,The views expressed in this information produc...
4,SRC01_P0004_C003,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,3,"If a translation of this work is created, it m..."


In [19]:
print("Total chunks:", len(chunks_df))

print("Duplicate chunk IDs:",
      chunks_df["chunk_id"].duplicated().sum())

print("Missing chunk IDs:",
      chunks_df["chunk_id"].isnull().sum())

print("Missing text:",
      chunks_df["text"].isnull().sum())

print("Empty chunks:",
      (chunks_df["text"].str.strip() == "").sum())

print("\nChunks per source:")
print(chunks_df.groupby("source_id").size())

print("\nChunk length statistics:")
print(chunks_df["text"].str.len().describe())

Total chunks: 5652
Duplicate chunk IDs: 0
Missing chunk IDs: 0
Missing text: 0
Empty chunks: 0

Chunks per source:
source_id
SRC01    1985
SRC02    3003
SRC03     573
SRC04      18
SRC05      73
dtype: int64

Chunk length statistics:
count    5652.000000
mean     1062.437721
std       314.554712
min         3.000000
25%       970.000000
50%      1100.000000
75%      1168.000000
max      4828.000000
Name: text, dtype: float64


In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully.")
print("Maximum sequence length:", embedding_model.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\shubham chavan\OneDrive\Documents\Desktop\biodiversity-intelligence-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shubham chavan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Maximum sequence length: 256


In [21]:
tokenizer = embedding_model.tokenizer

def get_token_count(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )
    return len(encoded["input_ids"])

chunks_df["token_count"] = chunks_df["text"].apply(get_token_count)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (269 > 256). Running this sequence through the model will result in indexing errors


In [22]:
print("Token count statistics:")
print(chunks_df["token_count"].describe())

too_long = chunks_df[
    chunks_df["token_count"] > embedding_model.max_seq_length
]

print("\nChunks exceeding 256 tokens:", len(too_long))

print(
    "Percentage exceeding limit:",
    round(len(too_long) / len(chunks_df) * 100, 2),
    "%"
)

print("Maximum token count:", chunks_df["token_count"].max())

Token count statistics:
count    5652.000000
mean      234.959837
std        85.767875
min         3.000000
25%       192.000000
50%       223.000000
75%       259.000000
max       901.000000
Name: token_count, dtype: float64

Chunks exceeding 256 tokens: 1462
Percentage exceeding limit: 25.87 %
Maximum token count: 901


In [23]:
tiny_chunks = chunks_df[
    chunks_df["token_count"] < 20
]

print("Chunks under 20 tokens:", len(tiny_chunks))

tiny_chunks[
    ["chunk_id", "source_id", "page_number", "token_count", "text"]
].head(20)

Chunks under 20 tokens: 33


,chunk_id,source_id,page_number,token_count,text
0,SRC01_P0001_C001,SRC01,1,16,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
40,SRC01_P0022_C001,SRC01,22,5,© Andy Murray
57,SRC01_P0030_C001,SRC01,30,5,© Andy Murray
69,SRC01_P0035_C001,SRC01,35,5,© Andy Murray
70,SRC01_P0036_C001,SRC01,36,12,©FAO / Matteo Sala © Andy Murray
278,SRC01_P0121_C001,SRC01,121,15,© Pavel Krásenský Global diversity and distrib...
360,SRC01_P0143_C001,SRC01,143,5,© Andy Murray
361,SRC01_P0144_C001,SRC01,144,5,© Andy Murray
393,SRC01_P0156_C001,SRC01,156,9,©FAO / Matteo Sala
611,SRC01_P0206_C001,SRC01,206,12,State of knowledge of soil biodiversity 176 © ...


In [24]:
def create_token_aware_chunks(
    text,
    tokenizer,
    target_tokens=220,
    overlap_sentences=2
):
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    def token_count(value):
        return len(
            tokenizer(
                value,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

    # --------------------------------------------------
    # Handle a single sentence/block that is too long
    # --------------------------------------------------
    expanded_sentences = []

    for sentence in sentences:

        if token_count(sentence) <= target_tokens:
            expanded_sentences.append(sentence)
            continue

        # Token-level fallback for oversized sentences,
        # TOCs, lists or badly extracted PDF blocks
        token_ids = tokenizer(
            sentence,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]

        # Leave room for special tokens
        safe_size = target_tokens - 2

        for start in range(0, len(token_ids), safe_size):

            token_piece = token_ids[start:start + safe_size]

            piece = tokenizer.decode(
                token_piece,
                skip_special_tokens=True
            ).strip()

            if piece:
                expanded_sentences.append(piece)

    # --------------------------------------------------
    # Pack sentences into token-safe chunks
    # --------------------------------------------------
    chunks = []
    current_sentences = []

    for sentence in expanded_sentences:

        candidate = " ".join(
            current_sentences + [sentence]
        ).strip()

        if (
            not current_sentences
            or token_count(candidate) <= target_tokens
        ):
            current_sentences.append(sentence)

        else:
            chunk = " ".join(current_sentences).strip()

            if chunk:
                chunks.append(chunk)

            # Keep previous sentences for context
            overlap = (
                current_sentences[-overlap_sentences:]
                if overlap_sentences > 0
                else []
            )

            # Make sure overlap itself doesn't make
            # the next chunk exceed the token target
            while overlap:
                candidate = " ".join(
                    overlap + [sentence]
                ).strip()

                if token_count(candidate) <= target_tokens:
                    break

                overlap = overlap[1:]

            current_sentences = overlap + [sentence]

    if current_sentences:
        final_chunk = " ".join(current_sentences).strip()

        if final_chunk:
            chunks.append(final_chunk)

    return chunks

In [25]:
test_chunks = create_token_aware_chunks(
    sample_text,
    tokenizer
)

print("Chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks, 1):

    tokens = len(
        tokenizer(
            chunk,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    print(
        f"Chunk {i}: "
        f"{len(chunk)} characters | "
        f"{tokens} tokens"
    )

Chunks: 7
Chunk 1: 755 characters | 197 tokens
Chunk 2: 820 characters | 206 tokens
Chunk 3: 1060 characters | 207 tokens
Chunk 4: 800 characters | 169 tokens
Chunk 5: 798 characters | 201 tokens
Chunk 6: 759 characters | 175 tokens
Chunk 7: 974 characters | 204 tokens


In [30]:
chunks_df["token_count"] = chunks_df["text"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

print("Token counts calculated successfully.")
print(chunks_df.columns.tolist())

Token counts calculated successfully.
['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text', 'token_count']


In [31]:
all_chunks = []

for _, row in retrieval_df.iterrows():

    page_chunks = create_token_aware_chunks(
        row["clean_text"],
        tokenizer,
        target_tokens=220,
        overlap_sentences=2
    )

    for chunk_index, chunk in enumerate(page_chunks, start=1):

        chunk_id = (
            f'{row["source_id"]}_'
            f'P{int(row["page_number"]):04d}_'
            f'C{chunk_index:03d}'
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "source_id": row["source_id"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)

print("New total chunks:", len(chunks_df))

New total chunks: 6876


In [34]:
print("Total chunks:", len(chunks_df))

print("\nIntegrity checks:")
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())
print("Missing IDs:", chunks_df["chunk_id"].isnull().sum())
print("Missing text:", chunks_df["text"].isnull().sum())
print("Empty chunks:", (chunks_df["text"].str.strip() == "").sum())

print("\nToken statistics:")
print(chunks_df["token_count"].describe())

print("\nChunks > 220:",
      (chunks_df["token_count"] > 220).sum())

print("Chunks > 256:",
      (chunks_df["token_count"] > 256).sum())

print("\nChunks per source:")
print(chunks_df.groupby("source_id").size())

Total chunks: 6876

Integrity checks:
Duplicate IDs: 0
Missing IDs: 0
Missing text: 0
Empty chunks: 0

Token statistics:


KeyError: 'token_count'

In [35]:
print(chunks_df.columns.tolist())

['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text']


In [36]:
def count_tokens(text):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

chunks_df["token_count"] = chunks_df["text"].apply(count_tokens)

print("Done!")
print(chunks_df.columns.tolist())

Done!
['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text', 'token_count']


In [37]:
chunks_df[["chunk_id", "token_count"]].head()

,chunk_id,token_count
0,SRC01_P0001_C001,16
1,SRC01_P0003_C001,27
2,SRC01_P0004_C001,208
3,SRC01_P0004_C002,209
4,SRC01_P0004_C003,187


In [38]:
print("Total chunks:", len(chunks_df))
print("Maximum tokens:", chunks_df["token_count"].max())

print(
    "Chunks > 220:",
    (chunks_df["token_count"] > 220).sum()
)

print(
    "Chunks > 256:",
    (chunks_df["token_count"] > 256).sum()
)

print("\nToken statistics:")
print(chunks_df["token_count"].describe())

Total chunks: 6876
Maximum tokens: 223
Chunks > 220: 8
Chunks > 256: 0

Token statistics:
count    6876.000000
mean      185.630745
std        39.050009
min         3.000000
25%       176.000000
50%       199.000000
75%       212.000000
max       223.000000
Name: token_count, dtype: float64


In [39]:
credit_mask = (
    chunks_df["text"].str.contains(
        r"^\s*©",
        regex=True,
        na=False
    )
    &
    (chunks_df["token_count"] < 20)
)

credit_chunks = chunks_df[credit_mask]

print("Obvious credit/noise chunks:", len(credit_chunks))

credit_chunks[
    ["chunk_id", "source_id", "page_number", "token_count", "text"]
]

Obvious credit/noise chunks: 18


,chunk_id,source_id,page_number,token_count,text
49,SRC01_P0022_C001,SRC01,22,5,© Andy Murray
71,SRC01_P0030_C001,SRC01,30,5,© Andy Murray
85,SRC01_P0035_C001,SRC01,35,5,© Andy Murray
86,SRC01_P0036_C001,SRC01,36,12,©FAO / Matteo Sala © Andy Murray
352,SRC01_P0121_C001,SRC01,121,15,© Pavel Krásenský Global diversity and distrib...
453,SRC01_P0143_C001,SRC01,143,5,© Andy Murray
454,SRC01_P0144_C001,SRC01,144,5,© Andy Murray
495,SRC01_P0156_C001,SRC01,156,9,©FAO / Matteo Sala
796,SRC01_P0220_C001,SRC01,220,8,©FAO/Ronald Vargas
878,SRC01_P0239_C001,SRC01,239,9,©FAO/Matteo Sala


In [40]:
before_count = len(chunks_df)

chunks_df = chunks_df[~credit_mask].copy()
chunks_df.reset_index(drop=True, inplace=True)

after_count = len(chunks_df)

print("Chunks before cleanup:", before_count)
print("Chunks after cleanup:", after_count)
print("Noise chunks removed:", before_count - after_count)

Chunks before cleanup: 6876
Chunks after cleanup: 6858
Noise chunks removed: 18


In [41]:
print("Final chunks:", len(chunks_df))

print("Duplicate IDs:",
      chunks_df["chunk_id"].duplicated().sum())

print("Missing text:",
      chunks_df["text"].isnull().sum())

print("Empty text:",
      (chunks_df["text"].str.strip() == "").sum())

print("Chunks > 256 tokens:",
      (chunks_df["token_count"] > 256).sum())

print("\nChunks per source:")
print(chunks_df.groupby("source_id").size())

Final chunks: 6858
Duplicate IDs: 0
Missing text: 0
Empty text: 0
Chunks > 256 tokens: 0

Chunks per source:
source_id
SRC01    2620
SRC02    3526
SRC03     589
SRC04      25
SRC05      98
dtype: int64


In [42]:
weird_short = chunks_df[
    chunks_df["token_count"] < 20
][
    ["chunk_id", "source_id", "page_number", "token_count", "text"]
]

weird_short

,chunk_id,source_id,page_number,token_count,text
0,SRC01_P0001_C001,SRC01,1,16,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
10,SRC01_P0005_C002,SRC01,5,17,soil biodiversity 98 2. 4. 1 | spatial pattern...
24,SRC01_P0012_C002,SRC01,12,11,responded to the survey 445 references 447
62,SRC01_P0027_C004,SRC01,27,15,of threatened species ld | land degradation ld...
66,SRC01_P0028_C004,SRC01,28,14,##cados pncti | technology and innovation plan
745,SRC01_P0206_C001,SRC01,206,12,State of knowledge of soil biodiversity 176 © ...
746,SRC01_P0207_C001,SRC01,207,12,Contributions of soil biodiversity to ecosyste...
747,SRC01_P0208_C001,SRC01,208,12,State of knowledge of soil biodiversity 178 © ...
748,SRC01_P0209_C001,SRC01,209,12,Contributions of soil biodiversity to ecosyste...
1035,SRC01_P0273_C004,SRC01,273,19,the highest rate of global warming and expansi...


### Chunk Quality Control

Obvious standalone copyright and image-credit fragments were removed from the
retrieval corpus. Short chunks were otherwise retained because some represent
meaningful section headings, captions, or scientific statements. Aggressive
length-based filtering was avoided to prevent accidental loss of valid
scientific context.

In [43]:
print("FINAL CHUNK CORPUS")
print("------------------")
print("Total chunks:", len(chunks_df))
print("Sources:", chunks_df["source_id"].nunique())
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())
print("Missing text:", chunks_df["text"].isnull().sum())
print("Empty text:", (chunks_df["text"].str.strip() == "").sum())
print("Maximum tokens:", chunks_df["token_count"].max())
print("Chunks exceeding MiniLM limit:",
      (chunks_df["token_count"] > embedding_model.max_seq_length).sum())

FINAL CHUNK CORPUS
------------------
Total chunks: 6858
Sources: 5
Duplicate IDs: 0
Missing text: 0
Empty text: 0
Maximum tokens: 223
Chunks exceeding MiniLM limit: 0


## 4. Generate Semantic Embeddings

Each validated scientific chunk is converted into a semantic vector using
the `all-MiniLM-L6-v2` sentence-transformer model.

These embeddings represent the meaning of each chunk and will allow
scientifically relevant passages to be retrieved based on semantic similarity
rather than exact keyword matching.

In [44]:
test_texts = chunks_df["text"].head(3).tolist()

test_embeddings = embedding_model.encode(
    test_texts,
    normalize_embeddings=True
)

print("Number of test embeddings:", len(test_embeddings))
print("Embedding dimensions:", test_embeddings.shape)

Number of test embeddings: 3
Embedding dimensions: (3, 384)


In [45]:
all_embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding generation complete.")
print("Embedding matrix shape:", all_embeddings.shape)

Batches:   0%|          | 0/215 [00:00<?, ?it/s]

Embedding generation complete.
Embedding matrix shape: (6858, 384)


In [46]:
import numpy as np

print("Number of chunks:", len(chunks_df))
print("Number of embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape[1])

print("Contains NaN:", np.isnan(all_embeddings).any())
print("Contains infinity:", np.isinf(all_embeddings).any())

Number of chunks: 6858
Number of embeddings: 6858
Embedding dimensions: 384
Contains NaN: False
Contains infinity: False


In [ ]:
## 5. Store Embeddings in ChromaDB

The validated scientific chunks and their MiniLM embeddings are stored in a
persistent ChromaDB collection.

Each vector retains its stable chunk identifier and provenance metadata,
including source document, filename, page number, and chunk index.

In [47]:
import chromadb

chroma_path = "../chroma_db"

client = chromadb.PersistentClient(
    path=chroma_path
)

print("ChromaDB client created.")

ChromaDB client created.


In [48]:
collection_name = "biodiversity_scientific_knowledge"

try:
    client.delete_collection(collection_name)
    print("Old collection removed.")
except Exception:
    print("No existing collection found.")

collection = client.create_collection(
    name=collection_name,
    metadata={
        "description": "Scientific biodiversity knowledge corpus",
        "embedding_model": "all-MiniLM-L6-v2",
        "chunking": "sentence-aware token-safe",
        "target_tokens": 220
    }
)

print("Fresh collection created.")

No existing collection found.
Fresh collection created.


In [49]:
ids = chunks_df["chunk_id"].tolist()
documents = chunks_df["text"].tolist()

metadatas = chunks_df[
    [
        "source_id",
        "filename",
        "page_number",
        "chunk_index"
    ]
].to_dict("records")

print("IDs:", len(ids))
print("Documents:", len(documents))
print("Metadata records:", len(metadatas))
print("Embeddings:", len(all_embeddings))

IDs: 6858
Documents: 6858
Metadata records: 6858
Embeddings: 6858


In [50]:
batch_size = 500

for start in range(0, len(chunks_df), batch_size):

    end = min(start + batch_size, len(chunks_df))

    collection.add(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
        embeddings=all_embeddings[start:end].tolist()
    )

    print(f"Stored {end}/{len(chunks_df)} chunks")

Stored 500/6858 chunks
Stored 1000/6858 chunks
Stored 1500/6858 chunks
Stored 2000/6858 chunks
Stored 2500/6858 chunks
Stored 3000/6858 chunks
Stored 3500/6858 chunks
Stored 4000/6858 chunks
Stored 4500/6858 chunks
Stored 5000/6858 chunks
Stored 5500/6858 chunks
Stored 6000/6858 chunks
Stored 6500/6858 chunks
Stored 6858/6858 chunks


In [51]:
print("Chunks in DataFrame:", len(chunks_df))
print("Chunks in ChromaDB:", collection.count())

assert collection.count() == len(chunks_df)

print("\nChromaDB validation passed.")

Chunks in DataFrame: 6858
Chunks in ChromaDB: 6858

ChromaDB validation passed.


## 6. Semantic Top-k Retrieval

A semantic retrieval function is implemented to convert a user query into a
MiniLM embedding and search ChromaDB for the most relevant scientific chunks.

The retriever returns the top-k matching chunks together with their stable
chunk IDs, source documents, page numbers, and similarity distances for
traceable scientific evidence retrieval.

In [52]:
def retrieve(query, top_k=3):

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(len(results["ids"][0])):

        retrieved_chunks.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved_chunks

In [53]:
query = "How does agricultural diversification affect biodiversity?"

results = retrieve(query, top_k=3)

for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("=" * 80)

    print("Chunk ID:", result["chunk_id"])
    print("Source:", result["metadata"]["source_id"])
    print("Page:", result["metadata"]["page_number"])
    print("Distance:", round(result["distance"], 4))

    print("\nText:")
    print(result["text"])


RESULT 1
Chunk ID: SRC05_P0002_C010
Source: SRC05
Page: 2
Distance: 0.4891

Text:
The most examined diversification practices were organic amendment, re- duced tillage, and crop diversification (146, 118, and 111 effect sizes), whereas noncrop diversification and inoculation, as well as organic farming, were less represented (38, 9, and 34 effect sizes) and need further investigations. Biodiversity and ecosystem service response to diversification The second-order meta-analysis showed that agricultural diversifi- cation strengthens several ecosystem service categories (omnibus test QM = 43.67; P < 0.0001; Fig. 2A) while having a neutral effect on crop yield [lnRR = 0.01; 95% confidence interval (CI) = −0.12 to 0.14].

RESULT 2
Chunk ID: SRC05_P0001_C008
Source: SRC05
Page: 1
Distance: 0.5092

Text:
Functional diversity below ground can also be supported and stimulated through addition of organic inputs (e.g., manure and crop residues) or reducing soil disturbance (e.g., reduced tillag

In [54]:
import pandas as pd

golden_df = pd.read_csv("../data/evaluation/golden_cases.csv")

golden_df

,case_id,question,known_soc,known_rainfall,known_land_use,known_biodiversity,missing_variables,required_knowledge,expected_behavior,forbidden_behavior
0,GC01,"A farm has low soil organic carbon, low rainfa...",0.3%,low,monoculture wheat,low species count,none,SOC <-> soil/ecosystem condition; SOC <-> biod...,Notice SOC + rainfall + land use + biodiversit...,Recommend generic 'plant more trees' without a...
1,GC02,"My land already has good soil carbon, decent r...",1.8%,adequate,agroforestry,high species count,none,Maintenance-level agroforestry/high-biodiversi...,Recognize this is a positive baseline; suggest...,Recommend a drastic intervention when conditio...
2,GC03,Biodiversity is declining on my land. Soil org...,0.4%,unknown,unknown,declining,"rainfall, land use",SOC <-> biodiversity decline indicators; what ...,Ask a targeted clarifying question for rainfal...,Fabricate rainfall or land use to produce an a...
3,GC04,We get decent rainfall but our soil carbon is ...,low,adequate,monoculture wheat,unknown,biodiversity indicator,SOC <-> monoculture interaction effects; how a...,Reason about SOC + land use with rainfall as a...,Assume biodiversity is low without evidence or...
4,GC05,What can I do to improve biodiversity on this ...,unknown,unknown,monoculture wheat,unknown,"soil organic carbon, rainfall, biodiversity in...",General monoculture <-> biodiversity evidence ...,Ask for the missing core variables (at minimum...,"Provide a fully specific, quantified recommend..."
5,GC06,"Rainfall here is high, but soil carbon is very...",very low,high,monoculture corn,moderate/stable,none,How high/non-limiting rainfall expands the ran...,Recognize that high rainfall removes the water...,Ignore rainfall as a relevant factor just beca...
6,GC07,"Semi-arid region, soil organic carbon 0.3%, ra...",0.3%,low,monoculture wheat,unknown,biodiversity indicator (optional),SOC <-> soil/ecosystem condition; rainfall (lo...,Recommend an evidence-supported intervention a...,Invent a quantified estimate or timeframe when...
7,GC08,What's the best fertilizer brand for my lawn?,unknown,unknown,lawn,unknown,not applicable - out of scope,Scope boundary definition for the system; how ...,Recognize the question falls outside the syste...,Fabricate an environmental recommendation for ...


## 7. Golden-Case Retrieval Evaluation

Retrieval quality is evaluated using the predefined Phase 1 golden cases.

For GC01–GC07, a case is considered a top-3 retrieval hit when at least one
of the three retrieved chunks contains scientific evidence relevant to the
case's predefined `required_knowledge`.

The criteria are fixed before inspecting retrieval results to avoid
post-hoc relevance judgments.

GC08 is evaluated separately as an out-of-scope case and is not included
in the scientific top-3 hit-rate calculation.

In [55]:
eval_df = golden_df[
    ["case_id", "question", "required_knowledge"]
].copy()

eval_df

,case_id,question,required_knowledge
0,GC01,"A farm has low soil organic carbon, low rainfa...",SOC <-> soil/ecosystem condition; SOC <-> biod...
1,GC02,"My land already has good soil carbon, decent r...",Maintenance-level agroforestry/high-biodiversi...
2,GC03,Biodiversity is declining on my land. Soil org...,SOC <-> biodiversity decline indicators; what ...
3,GC04,We get decent rainfall but our soil carbon is ...,SOC <-> monoculture interaction effects; how a...
4,GC05,What can I do to improve biodiversity on this ...,General monoculture <-> biodiversity evidence ...
5,GC06,"Rainfall here is high, but soil carbon is very...",How high/non-limiting rainfall expands the ran...
6,GC07,"Semi-arid region, soil organic carbon 0.3%, ra...",SOC <-> soil/ecosystem condition; rainfall (lo...
7,GC08,What's the best fertilizer brand for my lawn?,Scope boundary definition for the system; how ...


In [56]:
scientific_eval_df = eval_df[
    eval_df["case_id"] != "GC08"
].copy()

for _, row in scientific_eval_df.iterrows():
    print("=" * 80)
    print(row["case_id"])
    print("QUESTION:")
    print(row["question"])

    print("\nREQUIRED KNOWLEDGE:")
    print(row["required_knowledge"])
    print()

GC01
QUESTION:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

REQUIRED KNOWLEDGE:
SOC <-> soil/ecosystem condition; SOC <-> biodiversity; rainfall <-> vegetation establishment; rainfall <-> restoration constraints; monoculture <-> biodiversity; land-use diversification interventions; evidence on expected effects

GC02
QUESTION:
My land already has good soil carbon, decent rainfall, and agroforestry. Species counts are high. Is there anything more I should do?

REQUIRED KNOWLEDGE:
Maintenance-level agroforestry/high-biodiversity system evidence; SOC <-> biodiversity stability under good conditions; guidance on when no major intervention is warranted

GC03
QUESTION:
Biodiversity is declining on my land. Soil organic carbon is 0.4%.

REQUIRED KNOWLEDGE:
SOC <-> biodiversity decline indicators; what rainfall data is needed and why; what land-use data is needed and why; clarifying-question triggers

In [57]:
golden_retrieval_results = []

for _, row in scientific_eval_df.iterrows():

    results = retrieve(
        row["question"],
        top_k=3
    )

    for rank, result in enumerate(results, start=1):

        golden_retrieval_results.append({
            "case_id": row["case_id"],
            "question": row["question"],
            "required_knowledge": row["required_knowledge"],
            "rank": rank,
            "chunk_id": result["chunk_id"],
            "source_id": result["metadata"]["source_id"],
            "page_number": result["metadata"]["page_number"],
            "distance": result["distance"],
            "text": result["text"]
        })

golden_results_df = pd.DataFrame(golden_retrieval_results)

print(
    "Golden cases tested:",
    golden_results_df["case_id"].nunique()
)

print(
    "Total retrieved chunks:",
    len(golden_results_df)
)

Golden cases tested: 7
Total retrieved chunks: 21


In [58]:
for case_id in scientific_eval_df["case_id"]:

    case_results = golden_results_df[
        golden_results_df["case_id"] == case_id
    ]

    print("\n" + "=" * 100)
    print(case_id)
    print("=" * 100)

    print("\nQUESTION:")
    print(case_results.iloc[0]["question"])

    print("\nREQUIRED KNOWLEDGE:")
    print(case_results.iloc[0]["required_knowledge"])

    for _, result in case_results.iterrows():

        print("\n" + "-" * 80)

        print(
            f'RANK {result["rank"]} | '
            f'{result["chunk_id"]} | '
            f'{result["source_id"]} | '
            f'Page {result["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print("\nTEXT:")
        print(result["text"])


GC01

QUESTION:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

REQUIRED KNOWLEDGE:
SOC <-> soil/ecosystem condition; SOC <-> biodiversity; rainfall <-> vegetation establishment; rainfall <-> restoration constraints; monoculture <-> biodiversity; land-use diversification interventions; evidence on expected effects

--------------------------------------------------------------------------------
RANK 1 | SRC01_P0270_C005 | SRC01 | Page 270 | Distance 0.7656

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nut

In [59]:
for case_id in ["GC04", "GC07"]:

    case_results = golden_results_df[
        golden_results_df["case_id"] == case_id
    ]

    print("\n", "=" * 80)
    print(case_id)

    for _, row in case_results.iterrows():
        print(
            row["rank"],
            row["chunk_id"],
            row["source_id"],
            row["page_number"],
            round(row["distance"], 4)
        )


GC04
1 SRC01_P0601_C005 SRC01 601 0.865
2 SRC01_P0270_C005 SRC01 270 0.8686
3 SRC01_P0588_C001 SRC01 588 0.8877

GC07
1 SRC01_P0344_C001 SRC01 344 0.5801
2 SRC01_P0469_C001 SRC01 469 0.5877
3 SRC01_P0270_C005 SRC01 270 0.6326


In [60]:


for case_id in ["GC04", "GC07"]:

    question = scientific_eval_df.loc[
        scientific_eval_df["case_id"] == case_id,
        "question"
    ].iloc[0]

    diagnostic_results = retrieve(question, top_k=10)

    print("\n" + "=" * 100)
    print(case_id)
    print("=" * 100)

    for rank, result in enumerate(diagnostic_results, start=1):

        print(
            f'\nRANK {rank} | '
            f'{result["chunk_id"]} | '
            f'{result["metadata"]["source_id"]} | '
            f'Page {result["metadata"]["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print(result["text"][:500])


GC04

RANK 1 | SRC01_P0601_C005 | SRC01 | Page 601 | Distance 0.8650
2015. Soil fertility decline at the base of rural poverty in sub-Saharan Africa. Nature Plants, 1: 15101. doi: 10.1038/nplants.2015.101 Vargas-Rojas, R., Cuevas-Corona, R., Yigini, Y., Tong, Y., Bazza, Z. & Wiese, L. 2019. Unlocking the Potential of Soil Organic Carbon: A Feasible Way Forward. Pages 373–395 in T. Ginzky, H. Dooley, E. Heuser, I.L. Kasimbazi, E. Markus, T. Qin, eds. International Yearbook of Soil Law and Policy, 2018, pp. 373-395. Cham, Switzerland. Springer. Vasenev, V. & Kuzyak

RANK 2 | SRC01_P0270_C005 | SRC01 | Page 270 | Distance 0.8686
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion

In [61]:
diagnostic_queries = [
    "How does low soil organic carbon affect soil biodiversity and soil health?",
    "What restoration practices are suitable under low rainfall or semi-arid conditions?",
    "How can crop diversification, intercropping, or agroforestry improve biodiversity in monoculture farming?"
]

for query in diagnostic_queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    results = retrieve(query, top_k=3)

    for rank, result in enumerate(results, start=1):

        print(
            f'\nRANK {rank} | '
            f'{result["chunk_id"]} | '
            f'{result["metadata"]["source_id"]} | '
            f'Page {result["metadata"]["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print(result["text"][:600])


QUERY: How does low soil organic carbon affect soil biodiversity and soil health?

RANK 1 | SRC01_P0236_C003 | SRC01 | Page 236 | Distance 0.4231
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly impact soil biodiversity. For instance, soil biodiversity was higher in agricultural soils than in carbon-rich northern forests (Griffiths et al., 2016), but the main factor exp

RANK 2 | SRC01_P0227_C001 | SRC01 | Page 227 | Distance 0.5185
Threats to soil biodiversity - global and regional trends 197 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 C C C C C C C C C C C Impacts on soil biodiversity Erosion and landslides drivers 

### Multi-Query Retrieval

Complex user questions may contain several environmental factors whose relevant
evidence occurs in different scientific sources. A single query embedding can
underrepresent some of these factors.

The retriever therefore decomposes complex questions into focused evidence
queries covering soil organic carbon, rainfall/climate constraints, land-use
management, and biodiversity where applicable. Results are merged and
deduplicated using their stable chunk IDs.

In [67]:
def decompose_query(question):
    q = question.lower()

    queries = {
        "original": question
    }

    if any(term in q for term in [
        "soil carbon",
        "soil organic carbon",
        "soc"
    ]):
        queries["soc"] = (
            "How does soil organic carbon affect soil biodiversity "
            "and soil health?"
        )

    if any(term in q for term in [
        "rainfall",
        "semi-arid",
        "semi arid",
        "dryland",
        "drought"
    ]):
        queries["rainfall"] = (
            "How do rainfall and water availability affect "
            "restoration and vegetation establishment?"
        )

    if any(term in q for term in [
        "monoculture",
        "wheat",
        "corn",
        "maize",
        "agroforestry"
    ]):
        queries["land_use"] = (
            "How do agricultural land-use diversification, "
            "intercropping and agroforestry affect biodiversity?"
        )

    if any(term in q for term in [
        "biodiversity",
        "species",
        "species count",
        "species diversity"
    ]):
        queries["biodiversity"] = (
            "What agricultural management practices support "
            "biodiversity and ecosystem health?"
        )

    return queries

In [63]:
for case_id in ["GC01", "GC04", "GC07"]:

    question = scientific_eval_df.loc[
        scientific_eval_df["case_id"] == case_id,
        "question"
    ].iloc[0]

    print("\n", case_id)

    for query in decompose_query(question):
        print(" -", query)


 GC01
 - A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?
 - How does soil organic carbon affect soil biodiversity and soil health?
 - How do rainfall and water availability affect restoration and vegetation establishment?
 - How do agricultural land-use diversification, intercropping and agroforestry affect biodiversity?
 - What agricultural management practices support biodiversity and ecosystem health?

 GC04
 - We get decent rainfall but our soil carbon is low and we grow only wheat. What should we do?
 - How does soil organic carbon affect soil biodiversity and soil health?
 - How do rainfall and water availability affect restoration and vegetation establishment?
 - How do agricultural land-use diversification, intercropping and agroforestry affect biodiversity?

 GC07
 - Semi-arid region, soil organic carbon 0.3%, rainfall is low, crop is monoculture wheat. What should I do to improve biod

In [64]:
def retrieve_multi_query(question, top_k=3, per_query_k=5, rrf_k=60):

    queries = decompose_query(question)

    fused = {}

    for query in queries:

        results = retrieve(query, top_k=per_query_k)

        for rank, result in enumerate(results, start=1):

            chunk_id = result["chunk_id"]

            # Reciprocal Rank Fusion score
            score = 1 / (rrf_k + rank)

            if chunk_id not in fused:
                fused[chunk_id] = {
                    "chunk_id": chunk_id,
                    "text": result["text"],
                    "metadata": result["metadata"],
                    "rrf_score": 0.0,
                    "best_distance": result["distance"],
                    "matched_queries": []
                }

            fused[chunk_id]["rrf_score"] += score

            fused[chunk_id]["best_distance"] = min(
                fused[chunk_id]["best_distance"],
                result["distance"]
            )

            fused[chunk_id]["matched_queries"].append(query)

    ranked_results = sorted(
        fused.values(),
        key=lambda x: (
            -x["rrf_score"],
            x["best_distance"]
        )
    )

    return ranked_results[:top_k]

In [66]:
gc07_question = scientific_eval_df.loc[
    scientific_eval_df["case_id"] == "GC07",
    "question"
].iloc[0]

results = retrieve_multi_query(
    gc07_question,
    top_k=3
)

for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("=" * 80)

    print("Chunk ID:", result["chunk_id"])
    print("Source:", result["metadata"]["source_id"])
    print("Page:", result["metadata"]["page_number"])
    print("RRF Score:", round(result["rrf_score"], 6))
    print("Best Distance:", round(result["best_distance"], 4))
    print("Matched Queries:", len(result["matched_queries"]))

    print("\nText:")
    print(result["text"])


RESULT 1
Chunk ID: SRC01_P0236_C003
Source: SRC01
Page: 236
RRF Score: 0.016393
Best Distance: 0.4176
Matched Queries: 1

Text:
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly impact soil biodiversity. For instance, soil biodiversity was higher in agricultural soils than in carbon-rich northern forests (Griffiths et al., 2016), but the main factor explaining biodiversity was pH, and low pH soils tend to have higher carbon content. Several authors also highlight the importance of soil carbon quality in addition to quantity for below-ground diversity on a global scale (Crowther et al., 2019). For instance, Szoboszlay e

In [68]:
def retrieve_coverage_aware(question, top_k=3, per_aspect_k=3):
    queries = decompose_query(question)

    # We prefer specific evidence aspects over the original broad query
    aspect_priority = [
        "soc",
        "rainfall",
        "land_use",
        "biodiversity"
    ]

    selected = []
    selected_ids = set()

    # STEP 1:
    # Select the best unique result from each relevant aspect
    for aspect in aspect_priority:

        if aspect not in queries:
            continue

        results = retrieve(
            queries[aspect],
            top_k=per_aspect_k
        )

        for result in results:

            if result["chunk_id"] not in selected_ids:

                selected.append({
                    **result,
                    "retrieval_aspect": aspect
                })

                selected_ids.add(result["chunk_id"])
                break

        if len(selected) == top_k:
            return selected

    # STEP 2:
    # If fewer than top_k results were found,
    # use the original question to fill remaining slots
    if len(selected) < top_k:

        original_results = retrieve(
            queries["original"],
            top_k=per_aspect_k
        )

        for result in original_results:

            if result["chunk_id"] not in selected_ids:

                selected.append({
                    **result,
                    "retrieval_aspect": "original"
                })

                selected_ids.add(result["chunk_id"])

            if len(selected) == top_k:
                break

    return selected

In [69]:
gc07_question = scientific_eval_df.loc[
    scientific_eval_df["case_id"] == "GC07",
    "question"
].iloc[0]

results = retrieve_coverage_aware(
    gc07_question,
    top_k=3
)

for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("=" * 80)

    print("Aspect:", result["retrieval_aspect"])
    print("Chunk ID:", result["chunk_id"])
    print("Source:", result["metadata"]["source_id"])
    print("Page:", result["metadata"]["page_number"])
    print("Distance:", round(result["distance"], 4))

    print("\nText:")
    print(result["text"])


RESULT 1
Aspect: soc
Chunk ID: SRC01_P0236_C003
Source: SRC01
Page: 236
Distance: 0.4176

Text:
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly impact soil biodiversity. For instance, soil biodiversity was higher in agricultural soils than in carbon-rich northern forests (Griffiths et al., 2016), but the main factor explaining biodiversity was pH, and low pH soils tend to have higher carbon content. Several authors also highlight the importance of soil carbon quality in addition to quantity for below-ground diversity on a global scale (Crowther et al., 2019). For instance, Szoboszlay et al. (2017) found evidence of a

In [70]:
coverage_results = []

for _, row in scientific_eval_df.iterrows():

    results = retrieve_coverage_aware(
        row["question"],
        top_k=3
    )

    for rank, result in enumerate(results, start=1):

        coverage_results.append({
            "case_id": row["case_id"],
            "question": row["question"],
            "required_knowledge": row["required_knowledge"],
            "rank": rank,
            "aspect": result["retrieval_aspect"],
            "chunk_id": result["chunk_id"],
            "source_id": result["metadata"]["source_id"],
            "page_number": result["metadata"]["page_number"],
            "distance": result["distance"],
            "text": result["text"]
        })

coverage_results_df = pd.DataFrame(coverage_results)

print(
    "Golden cases tested:",
    coverage_results_df["case_id"].nunique()
)

print(
    "Total retrieved chunks:",
    len(coverage_results_df)
)

Golden cases tested: 7
Total retrieved chunks: 21


In [71]:
for case_id in scientific_eval_df["case_id"]:

    case_results = coverage_results_df[
        coverage_results_df["case_id"] == case_id
    ]

    print("\n" + "=" * 100)
    print(case_id)
    print("=" * 100)

    print("\nQUESTION:")
    print(case_results.iloc[0]["question"])

    print("\nREQUIRED KNOWLEDGE:")
    print(case_results.iloc[0]["required_knowledge"])

    for _, result in case_results.iterrows():

        print("\n" + "-" * 80)

        print(
            f'RANK {result["rank"]} | '
            f'Aspect: {result["aspect"]} | '
            f'{result["chunk_id"]} | '
            f'{result["source_id"]} | '
            f'Page {result["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print("\nTEXT:")
        print(result["text"])


GC01

QUESTION:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

REQUIRED KNOWLEDGE:
SOC <-> soil/ecosystem condition; SOC <-> biodiversity; rainfall <-> vegetation establishment; rainfall <-> restoration constraints; monoculture <-> biodiversity; land-use diversification interventions; evidence on expected effects

--------------------------------------------------------------------------------
RANK 1 | Aspect: soc | SRC01_P0236_C003 | SRC01 | Page 236 | Distance 0.4176

TEXT:
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly i